## Introduction to NetworkX

![](img/NetworkX.jpg)

---

### Get prepared

#### Required installations

tbd

#### Imports

In [ ]:
import networkx as nx
import pandas as pd
pd.options.display.max_rows = 400
import matplotlib.pyplot as plt

#### Some functions

In [ ]:
def plot_graph(G, size=5):
    plt.figure(figsize=(size, size))
    nx.draw(G, with_labels=True, node_color='skyblue', width=.3, font_size=8)
    plt.show()

def plot_graph_with_weights(G):
    pos = nx.planar_layout(G) # pos = nx.nx_agraph.graphviz_layout(G)
    nx.draw_networkx(G,pos)
    labels = nx.get_edge_attributes(G,'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)
    plt.show()

---

### Understanding a graph's basic properties


#### Create an empty graph

In [ ]:
G = nx.Graph()
type(G)

#### Populate the graph.


 We can add one node at time using the `add-node` method or add any iterable collections (lists, strings etc..) with the `G.add_nodes_from` method. 

We are now using strings as nodes's names, but nodes can be any hashable object.

In [ ]:
G.add_node('a')
my_list = ['b', 'c', 'd']
G.add_nodes_from(my_list)

We can __inspect the nodes__ in a graph using *G.nodes()*.

In [ ]:
G.nodes()
list(G.nodes())

In [ ]:
plot_graph(G)

Now let's add edges to the graph. 

We can add one edge at time with *G.add_edge()*, specifying the nodes between whom we want the edge to stay. As for nodes, we can pass an iterable colletion to *G.add_edges_from()*.

In [ ]:
G.add_edges_from([('a', 'b'), ('a', 'c'), ('c', 'd'), ('b', 'd')])
G.edges()
list(G.edges())

If we add edges between nodes that do not exist, __theses nodes are automatically added__.

In [ ]:
G.add_edge("a", "e")
G.add_edge("a", "b")

In [ ]:
# Number of nodes
print("Number of Nodes: ", G.order()) # equivalent to len(G) and G.number_of_nodes()
# Number of edges
print("Number of Edges: ", G.number_of_edges())

In [ ]:
plot_graph(G, 12)

__Remove__ nodes using the *G.remove_node()* or the *G.remove_nodes_from()*

In [ ]:
G.remove_node('e')
plot_graph(G)

#### Querying node information

Let's now query for the nodeset. See the first 5 nodes.
(We use the `list()` constructor to refer to the Python basic types.)

In [ ]:
list(G.nodes())[0:5]

Because a `G.nodes()` is iterable, 
we can query it for its length:

In [ ]:
len(G.nodes())

#### Querying edge information

In [ ]:
list(G.edges())[0:5]

---

### Add attributes to nodes

In [30]:
for node in G.nodes:
    G.nodes[node]['color'] = 'white'

In [31]:
list(G.nodes(data=True))

[('a', {'degree': 3, 'color': 'white'}),
 ('b', {'degree': 2, 'color': 'white'}),
 ('c', {'degree': 2, 'color': 'white'}),
 ('d', {'degree': 2, 'color': 'white'}),
 ('e', {'degree': 1, 'color': 'white'})]

In [32]:
G.nodes['a']['color']

'white'

Additionally, we can set the attributes of individual nodes:

In [33]:
G.nodes["c"]['color'] = 'green'
G.nodes["d"]['color'] = 'yellow'
list(G.nodes(data=True))

[('a', {'degree': 3, 'color': 'white'}),
 ('b', {'degree': 2, 'color': 'white'}),
 ('c', {'degree': 2, 'color': 'green'}),
 ('d', {'degree': 2, 'color': 'yellow'}),
 ('e', {'degree': 1, 'color': 'white'})]

---

### Add attributes to edges

In [34]:
G['a']['b']['color'] = 'green'
G['a']['c']['color'] = 'black'
list(G.edges(data=True))

[('a', 'b', {'color': 'green'}),
 ('a', 'c', {'color': 'black'}),
 ('a', 'e', {}),
 ('b', 'd', {}),
 ('c', 'd', {})]

---

### Some calculations with graph measures

#### Calculate node degree

Make a Pandas dataframe from the degree data `G.nodes(data='degree')`, then sort from highest to lowest

In [ ]:
#print (G.nodes(data='degree'))
#for x in list(G.nodes(data='degree'))[0:5]:
#    print (type(x))
degree_df = pd.DataFrame(G.nodes(data='degree'), columns=['node', 'degree'])
degree_df = degree_df.sort_values(by='degree', ascending=False)
degree_df

Plot the nodes with the highest degree values

In [ ]:
num_nodes_to_inspect = 10
degree_df[:num_nodes_to_inspect].plot(x='node', y='degree', kind='barh').invert_yaxis()

Who has the most number of connections in the network?

In [ ]:
deg = dict(nx.degree(G))
max_key = max(deg, key=deg.get)
max_key

Make the degree values a `dict`ionary, then add it as a network "attribute" with `networkx.set_node_attributes()`

In [ ]:
degrees = dict(nx.degree(G))
nx.set_node_attributes(G, name='degree', values=degrees)
dict(G.nodes(data='degree'))

Now, we can write things like:

In [ ]:
for n, attr in list(G.nodes(data=True))[0:5]:
    # n is the node
    # attr is the attributes dictionary
    print(f"{n} {attr}")

#### Calculate Weighted Degree

Who has the most number of connections in the network (if you factor in edge weight)?

In [ ]:
nx.degree(G, weight='Weight')

Make the weighted degree values a `dict`ionary, then add it as a network "attribute" with `networkx.set_node_attributes()`

In [ ]:
weighted_degrees = dict(nx.degree(G, weight='Weight'))
nx.set_node_attributes(G, name='weighted_degree', values=weighted_degrees)

Make a Pandas dataframe from the degree data `G.nodes(data='weighted_degree')`, then sort from highest to lowest

In [ ]:
weighted_degree_df = pd.DataFrame(G.nodes(data='weighted_degree'), columns=['node', 'weighted_degree'])
weighted_degree_df = weighted_degree_df.sort_values(by='weighted_degree', ascending=False)
weighted_degree_df

Plot the nodes with the highest weighted degree values

In [ ]:
num_nodes_to_inspect = 10
weighted_degree_df[:num_nodes_to_inspect].plot(x='node', y='weighted_degree', color='orange', kind='barh').invert_yaxis()

#### Calculate Betweenness Centrality Scores

In graph theory, betweenness centrality is a measure of centrality in a graph based on shortest paths. 
In short: How many shortest paths pass a node?
More precise: What is the fraction of a graph's shortest paths that pass a certain node?

In [ ]:
nx.betweenness_centrality(G)

In [ ]:
betweenness_centrality = nx.betweenness_centrality(G)

Add `betweenness_centrality` (which is already a dictionary) as a node attribute with `networkx.set_node_attributes()`

In [ ]:
nx.set_node_attributes(G, name='betweenness', values=betweenness_centrality)

Make a Pandas dataframe from the betweenness data `G.nodes(data='betweenness')`, then sort from highest to lowest

In [ ]:
betweenness_df = pd.DataFrame(G.nodes(data='betweenness'), columns=['node', 'betweenness'])
betweenness_df = betweenness_df.sort_values(by='betweenness', ascending=False)
betweenness_df

Plot the nodes with the highest betweenness centrality scores

In [ ]:
num_nodes_to_inspect = 10
betweenness_df[:num_nodes_to_inspect].plot(x='node', y='betweenness', color='green', kind='barh').invert_yaxis()

---

### Serialize and deserialize a graph to/from JSON

We export our graph G to a json-coded file. JSON is similar to a Python dict. The graph dict has the following keys: "directed" (true/false), "multigraph" (true/false), "graph" (dict of global graph attributes), "nodes", and "edges".

In [ ]:
from pprint import pprint
import json
# Serialize to node-link format
with open('graphs/GOT_graph.json', 'w', encoding='utf-8') as f:
    json.dump(nx.node_link_data(G), f, indent=2, ensure_ascii=False)
# Deserialize from node-link format
with open('graphs/GOT_graph.json', 'r') as f:
    nl_data = json.load(f)
# Print a few nodes and edges
pprint(nl_data["nodes"][10:15])
pprint(nl_data["edges"][10:15])
# Reconstruct the graph
H = nx.node_link_graph(nl_data)
print(H)